# Насколько SFT размазывает распределение следующего токена (EXP-021)

Гипотеза после EXP-019 и EXP-020: SFT на текстах сильных моделей делает распределение следующего токена шире
(модель «неувереннее»), и выборка без top-k при T 0.9 чаще берёт токены из хвоста — ответ сходит с рельсов.
Проверка — без генерации: на готовых текстах считаем по каждому токену ответа (teacher forcing, тот же промпт, что при
генерации, режим без поиска):

- энтропию распределения следующего токена при T 0.9 (как при выборке) и при T 1;
- долю вероятности вне 20 самых вероятных токенов при T 0.9 — вероятность того, что выборка без top-k возьмёт токен,
  который top-k 20 отрезал бы;
- −log p фактического токена (при T 1).

Модели: исходная T-lite и адаптеры эпох 1, 4 и 6 из EXP-020 (`runs/sft/20261111_2313/epochs/`). Тексты: ответы
исходной T-lite (EXP-019, T 0.9), ответы эпохи 6 (EXP-020, T 0.9) — без поиска, номера 0 и 1 на 25 темах, только
невырожденные; 9 отложенных примеров SFT (тексты составителей).

Правило (до запуска, `docs/decisions.md`, раздел 9). Гипотеза подтверждается, если на ответах исходной модели
и на ответах эпохи 6 у эпохи 6 выше, чем у исходной, и энтропия при T 0.9, и доля вероятности вне top-20: разница
по одним и тем же текстам с 95% интервалом бутстрепа по текстам целиком выше нуля — все четыре условия. Дополнительно
(не условие): у эпохи 1 эффект больше, чем у эпохи 6 — как у доли вырожденных ответов.

Карта 0, около 40 минут (загрузка модели ~13 мин). Результат — `runs/entropy_check/20261113/`.

In [ ]:
import json, os, random, statistics, sys, time
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))
import torch
from peft import PeftModel

from presentation_rl.environment import END
from presentation_rl.evaluation import degeneration, write_csv
from presentation_rl.model_loader import load_model
from presentation_rl.tasks import load_tasks
from presentation_rl.training import sft
from presentation_rl.training.common import backbone_and_head, episode_prompt, mode_task, target_positions

OUT = ROOT / "runs" / "entropy_check" / "20261113"
OUT.mkdir(parents=True, exist_ok=True)
SFT_RUN = ROOT / "runs" / "sft" / "20261111_2313"
ADAPTERS = {"e1": SFT_RUN / "epochs" / "epoch_1", "e4": SFT_RUN / "epochs" / "epoch_4", "e6": SFT_RUN / "epochs" / "epoch_6"}
ANSWERS = {"base_answers": ROOT / "runs" / "sft_select" / "20261111" / "base_T09",
           "e6_answers": ROOT / "runs" / "sft_select" / "20261112_exp020" / "e6_T09"}
T, TOP_K, CHUNK = 0.9, 20, 512

start = time.time()
model, tokenizer = load_model("t-tech/T-lite-it-2.1", "main", dtype="float16", device_map={"": 0})
model = PeftModel.from_pretrained(model, ADAPTERS["e6"], adapter_name="e6")
for name in ("e1", "e4"):
    model.load_adapter(ADAPTERS[name], adapter_name=name)
model.eval()
print(f"загрузка {time.time() - start:.0f} с, память {torch.cuda.memory_allocated() / 2**30:.1f} ГБ")

In [ ]:
validation = {t.id: t for t in load_tasks(ROOT / "data" / "tasks" / "tasks.json", split="validation")}
sft_tasks = {t.id: t for t in load_tasks(ROOT / "data" / "tasks" / "tasks.json", split="sft")}
prompt = episode_prompt(tokenizer)


def item(request, answer):
    prompt_ids = prompt(request)
    answer_ids = tokenizer.encode(answer + END, add_special_tokens=False)
    return prompt_ids + answer_ids, [0] * len(prompt_ids) + [1] * len(answer_ids)


TEXTS = {}  # набор → [(id текста, ids, mask)]
for name, folder in ANSWERS.items():
    TEXTS[name] = []
    for key in sorted(validation):
        for s in (0, 1):
            meta = json.loads((folder / f"plain_{key}_{s}.json").read_text(encoding="utf-8"))
            answer = (folder / f"plain_{key}_{s}_answer.txt").read_text(encoding="utf-8")
            if meta.get("error") or degeneration(answer, meta["finish"])["degenerate"]:
                continue
            TEXTS[name].append((f"{key}_{s}", *item(mode_task(validation[key], "plain").request, answer)))
config = sft.SFTConfig.load(ROOT / "configs" / "experiments" / "sft_tlite_v2.yaml")
TEXTS["sft_heldout"] = []
for line in (ROOT / sft.PLAIN).read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    if record["id"] in config.eval_topics:
        TEXTS["sft_heldout"].append((record["id"], *item(sft.request_for(record, sft_tasks), record["answer"])))
for name, items in TEXTS.items():
    print(name, "текстов", len(items), "токенов ответа (медиана)", statistics.median(sum(m) for _, _, m in items))

## Замер

По каждому тексту и модели — средние по токенам ответа. Исходная модель — те же веса с выключенным адаптером.

In [ ]:
@torch.no_grad()
def measure(ids, mask):
    body, head = backbone_and_head(model)
    device = next(model.parameters()).device
    input_ids = torch.tensor([ids], device=device)
    positions = torch.tensor([p - 1 for p in target_positions(mask)], device=device)
    hidden = body(input_ids=input_ids, use_cache=False).last_hidden_state[0].index_select(0, positions)
    targets = input_ids[0].index_select(0, positions + 1)
    sums = {"entropy_T09": 0.0, "entropy_T1": 0.0, "tail_top20_T09": 0.0, "nll_T1": 0.0}
    for start in range(0, len(positions), CHUNK):
        logits = head(hidden[start:start + CHUNK]).float()
        for temp, key in ((T, "T09"), (1.0, "T1")):
            logp = torch.log_softmax(logits / temp, dim=-1)
            p = logp.exp()
            sums[f"entropy_{key}"] += float(-(p * logp).sum(-1).sum())
            if key == "T09":
                sums["tail_top20_T09"] += float((1 - p.topk(TOP_K, dim=-1).values.sum(-1)).sum())
            else:
                sums["nll_T1"] += float(-logp.gather(1, targets[start:start + CHUNK, None]).sum())
        del logits, logp, p
    n = len(positions)
    return {"tokens": n, **{k: v / n for k, v in sums.items()}}


rows = []
started = time.time()
for model_name in ("base", "e1", "e4", "e6"):
    for set_name, items in TEXTS.items():
        for text_id, ids, mask in items:
            if model_name == "base":
                with model.disable_adapter():
                    stats = measure(ids, mask)
            else:
                model.set_adapter(model_name)
                stats = measure(ids, mask)
            rows.append({"model": model_name, "texts": set_name, "id": text_id, **stats})
    print(f"{model_name}: готово, {(time.time() - started) / 60:.0f} мин", flush=True)
write_csv(OUT / "entropy.csv", rows)

## Итог и правило

In [ ]:
METRICS = ("entropy_T09", "tail_top20_T09", "entropy_T1", "nll_T1")


def mean(model_name, set_name, metric):
    return statistics.mean(r[metric] for r in rows if r["model"] == model_name and r["texts"] == set_name)


def paired(set_name, metric, other, base="base", n_boot=2000, seed=0):
    get = {(r["model"], r["id"]): r[metric] for r in rows if r["texts"] == set_name}
    ids = sorted({r["id"] for r in rows if r["texts"] == set_name})
    diffs = [get[(other, i)] - get[(base, i)] for i in ids]
    rng = random.Random(seed)
    boot = sorted(statistics.mean(rng.choice(diffs) for _ in diffs) for _ in range(n_boot))
    return statistics.mean(diffs), boot[int(0.025 * n_boot)], boot[int(0.975 * n_boot)]


summary = []
for set_name in TEXTS:
    print(f"\n{set_name} (текстов {len(TEXTS[set_name])})")
    for model_name in ("base", "e1", "e4", "e6"):
        row = {"texts": set_name, "model": model_name, **{m: mean(model_name, set_name, m) for m in METRICS}}
        summary.append(row)
        print(f"  {model_name:4s} " + "  ".join(f"{m} {row[m]:.4f}" for m in METRICS))
    for other in ("e1", "e4", "e6"):
        print(f"  {other} − base: " + "  ".join(
            "{} {:+.4f} [{:+.4f}; {:+.4f}]".format(m, *paired(set_name, m, other)) for m in ("entropy_T09", "tail_top20_T09")))
write_csv(OUT / "summary.csv", summary)

checks = {f"{s}/{m}": paired(s, m, "e6")[1] > 0 for s in ("base_answers", "e6_answers") for m in ("entropy_T09", "tail_top20_T09")}
print("\nусловия правила (нижняя граница интервала e6 − base > 0):", checks)
print("Вывод по правилу:", "гипотеза подтверждается" if all(checks.values()) else "гипотеза не подтверждается")
lengths = {s: statistics.median(sum(m) for _, _, m in TEXTS[s]) for s in ("base_answers", "e6_answers")}
for model_name in ("base", "e6"):
    s = "base_answers" if model_name == "base" else "e6_answers"
    print(f"{model_name}: ожидаемое число токенов из хвоста за top-20 на ответ (своя медиана длины {lengths[s]:.0f}) ≈ "
          f"{mean(model_name, s, 'tail_top20_T09') * lengths[s]:.1f}")
(OUT / "summary.json").write_text(json.dumps({"checks": checks, "confirmed": all(checks.values())}, ensure_ascii=False, indent=1),
                                  encoding="utf-8")